# Pull CVE Data from the NVD API

Run each cell in order. Before running the fetch cell, get a free API key here (takes about a minute):

https://nvd.nist.gov/developers/request-an-api-key

Without a key it still works, just slower (5 requests per 30 seconds instead of 50).

In [1]:
%pip install requests pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
import time
import requests
import pandas as pd

## Settings

Paste your API key between the quotes below if you have one. If you don't have one yet, leave it as `None` - it'll just run slower.

In [5]:
API_KEY = "32CC5337-59AD-46DB-9B40-72A37B282F71"

PUB_START_DATE = "2026-06-01T00:00:00.000"
PUB_END_DATE = "2026-09-18T00:00:00.000"

RESULTS_PER_PAGE = 200
API_URL = "https://services.nvd.nist.gov/rest/json/cves/2.0"

## Fetch the data

This will print progress as it goes (NVD only returns a limited number of records per request, so this loops until it has them all). This can take a few minutes depending on the date range and whether you have an API key.

In [6]:
headers = {"apiKey": API_KEY} if API_KEY else {}
all_records = []
start_index = 0

while True:
    params = {
        "pubStartDate": PUB_START_DATE,
        "pubEndDate": PUB_END_DATE,
        "resultsPerPage": RESULTS_PER_PAGE,
        "startIndex": start_index,
    }
    response = requests.get(API_URL, params=params, headers=headers)
    response.raise_for_status()
    data = response.json()

    vulnerabilities = data.get("vulnerabilities", [])
    if not vulnerabilities:
        break

    for item in vulnerabilities:
        cve = item["cve"]
        cve_id = cve["id"]
        published = cve.get("published", "")

        description = ""
        for d in cve.get("descriptions", []):
            if d.get("lang") == "en":
                description = d.get("value", "")
                break

        cvss_score, cvss_severity, cvss_version = None, None, None
        metrics = cve.get("metrics", {})
        for key in ("cvssMetricV31", "cvssMetricV30", "cvssMetricV2"):
            if key in metrics and metrics[key]:
                m = metrics[key][0]
                cvss_score = m["cvssData"].get("baseScore")
                cvss_severity = m.get("baseSeverity", m["cvssData"].get("baseSeverity"))
                cvss_version = key
                break

        weaknesses = []
        for w in cve.get("weaknesses", []):
            for d in w.get("description", []):
                if d.get("lang") == "en":
                    weaknesses.append(d.get("value"))
        cwe = ";".join(weaknesses)

        all_records.append({
            "cve_id": cve_id,
            "published": published,
            "description": description,
            "cvss_score": cvss_score,
            "cvss_severity": cvss_severity,
            "cvss_version": cvss_version,
            "cwe": cwe,
        })

    total_results = data.get("totalResults", 0)
    start_index += RESULTS_PER_PAGE
    print(f"Fetched {min(start_index, total_results)} / {total_results} CVEs...")

    if start_index >= total_results:
        break

    time.sleep(6 if not API_KEY else 0.7)

cve_df = pd.DataFrame(all_records)
print(f"\nDone. {len(cve_df)} CVE records fetched.")

Fetched 200 / 40519 CVEs...
Fetched 400 / 40519 CVEs...
Fetched 600 / 40519 CVEs...
Fetched 800 / 40519 CVEs...
Fetched 1000 / 40519 CVEs...
Fetched 1200 / 40519 CVEs...
Fetched 1400 / 40519 CVEs...
Fetched 1600 / 40519 CVEs...
Fetched 1800 / 40519 CVEs...
Fetched 2000 / 40519 CVEs...
Fetched 2200 / 40519 CVEs...
Fetched 2400 / 40519 CVEs...
Fetched 2600 / 40519 CVEs...
Fetched 2800 / 40519 CVEs...
Fetched 3000 / 40519 CVEs...
Fetched 3200 / 40519 CVEs...
Fetched 3400 / 40519 CVEs...
Fetched 3600 / 40519 CVEs...
Fetched 3800 / 40519 CVEs...
Fetched 4000 / 40519 CVEs...
Fetched 4200 / 40519 CVEs...
Fetched 4400 / 40519 CVEs...
Fetched 4600 / 40519 CVEs...
Fetched 4800 / 40519 CVEs...
Fetched 5000 / 40519 CVEs...
Fetched 5200 / 40519 CVEs...
Fetched 5400 / 40519 CVEs...
Fetched 5600 / 40519 CVEs...
Fetched 5800 / 40519 CVEs...
Fetched 6000 / 40519 CVEs...
Fetched 6200 / 40519 CVEs...
Fetched 6400 / 40519 CVEs...
Fetched 6600 / 40519 CVEs...
Fetched 6800 / 40519 CVEs...
Fetched 7000 / 405

## Look at what we got

In [7]:
cve_df.head()

,cve_id,published,description,cvss_score,cvss_severity,cvss_version,cwe
0,CVE-2026-10201,2026-06-01T00:16:41.927,A vulnerability was determined in Assimp up to...,3.3,LOW,cvssMetricV31,CWE-369;CWE-404
1,CVE-2026-10202,2026-06-01T00:16:42.097,A vulnerability was identified in OFCMS 1.1.3....,6.3,MEDIUM,cvssMetricV31,CWE-74;CWE-89
2,CVE-2026-10203,2026-06-01T00:16:42.257,A security flaw has been discovered in OFCMS 1...,6.3,MEDIUM,cvssMetricV31,CWE-74;CWE-89
3,CVE-2026-10204,2026-06-01T00:16:42.427,A weakness has been identified in OFCMS 1.1.3....,6.3,MEDIUM,cvssMetricV31,CWE-74;CWE-89
4,CVE-2026-10205,2026-06-01T01:16:47.450,A security vulnerability has been detected in ...,6.3,MEDIUM,cvssMetricV31,CWE-284;CWE-434


In [8]:
cve_df['cvss_severity'].value_counts()

cvss_severity
HIGH        16056
MEDIUM      13056
CRITICAL     4599
LOW          1338
NONE           10
Name: count, dtype: int64

## Save it

In [9]:
cve_df.to_csv("cve_raw.csv", index=False)
print("Saved cve_raw.csv")

Saved cve_raw.csv
